In [ ]:
import subprocess
import sys
import os

print("=" * 60)
print("Checking and installing required packages...")
print("=" * 60)

required_packages = [
    'transformers>=4.36.0',
    'torch',
    'accelerate',
    'pandas',
    'numpy',
    'scikit-learn',
    'huggingface_hub',
    'tqdm',
    'scipy',
    'matplotlib',
    'seaborn'
]

for package in required_packages:
    package_name = package.split('>=')[0] if '>=' in package else package
    try:
        __import__(package_name)
        print(f"✓ {package_name} is already installed")
    except ImportError:
        print(f"✗ Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import warnings
import gc
import matplotlib.pyplot as plt
import seaborn as sns
import json
from datetime import datetime
from huggingface_hub import login

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def authenticate_huggingface():
    print("\n" + "=" * 60)
    print("Hugging Face Authentication")
    print("=" * 60)

    token = None
    token_path = os.path.expanduser("~/.huggingface/token")

    if os.path.exists(token_path):
        with open(token_path, 'r') as f:
            token = f.read().strip()
        print("✓ Using existing Hugging Face token")
    else:
        print("\nPlease enter your Hugging Face access token.")
        print("Get it from: https://huggingface.co/settings/tokens")
        token = input("Enter token: ").strip()
        
        os.makedirs(os.path.dirname(token_path), exist_ok=True)
        with open(token_path, 'w') as f:
            f.write(token)

    try:
        login(token=token)
        print("✓ Logged in to Hugging Face")
        return token
    except Exception as e:
        print(f"✗ Login error: {e}")
        return None

class PlagiarismDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.codes = df["clean_code"].fillna("").astype(str).tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.codes)
    
    def __getitem__(self, idx):
        code = self.codes[idx]
        label = self.labels[idx]
        
        inputs = self.tokenizer(
            code,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        
        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

class CodeLlamaForPlagiarismDetection(nn.Module):
    def __init__(self, model_name="codellama/CodeLlama-7b-hf", token=None):
        super().__init__()
        
        print(f"Loading {model_name}...")
        
        self.codellama = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            token=token,
            trust_remote_code=True
        )
        
        print("✓ Model loaded successfully!")
        
        self.hidden_size = self.codellama.config.hidden_size
        print(f"  Hidden size: {self.hidden_size}")
        
        for param in self.codellama.parameters():
            param.requires_grad = False
        
        self.classifier = nn.Sequential(
            nn.Linear(self.hidden_size, 512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 2)
        )
        
        for layer in self.classifier:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)
        
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.classifier.parameters())
        
        print(f"  Total parameters: {total_params:,}")
        print(f"  Trainable parameters: {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")
    
    def forward(self, input_ids, attention_mask):
        if self.codellama.device != input_ids.device:
            self.codellama = self.codellama.to(input_ids.device)
        
        with torch.no_grad():
            outputs = self.codellama(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
        
        last_hidden_state = outputs.last_hidden_state
        
        attention_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * attention_mask_expanded, dim=1)
        sum_mask = torch.clamp(attention_mask_expanded.sum(dim=1), min=1e-9)
        pooled_output = sum_embeddings / sum_mask
        
        logits = self.classifier(pooled_output)
        return logits

def train_codellama(model, train_loader, optimizer, criterion, device, epochs, seed):
    model.train()
    
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0
        
        pbar = tqdm(train_loader, desc=f"CodeLlama - Epoch {epoch+1}/{epochs} (Seed {seed})")
        for batch in pbar:
            optimizer.zero_grad()
            
            logits = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device)
            )
            
            loss = criterion(logits, batch["labels"].to(device))
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.classifier.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_loss += loss.item()
            
            _, predicted = torch.max(logits, 1)
            batch_total = batch["labels"].size(0)
            batch_correct = (predicted.cpu() == batch["labels"]).sum().item()
            
            total += batch_total
            correct += batch_correct
            
            avg_loss = total_loss / (pbar.n + 1)
            pbar.set_postfix({'loss': f'{avg_loss:.4f}', 'acc': f'{100*batch_correct/batch_total:.2f}%'})
        
        epoch_acc = 100 * correct / total if total > 0 else 0
        avg_loss = total_loss / len(train_loader) if len(train_loader) > 0 else 0
        print(f"[Seed {seed}, CodeLlama] Epoch {epoch+1} Loss: {avg_loss:.4f}, Acc: {epoch_acc:.2f}%")
    
    return model

def evaluate_model(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating"):
            logits = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device)
            )
            preds = torch.argmax(logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch["labels"].cpu().numpy())
    
    if len(all_labels) > 0:
        accuracy = accuracy_score(all_labels, all_preds)
        precision = precision_score(all_labels, all_preds, average='binary', zero_division=0)
        recall = recall_score(all_labels, all_preds, average='binary', zero_division=0)
        f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    else:
        accuracy = precision = recall = f1 = 0.0
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'predictions': all_preds,
        'labels': all_labels
    }

def evaluate_all_test_files(model, tokenizer, device, seed):
    results = {}
    
    for i in range(10):
        test_file = f"Test_{i}.csv"
        if os.path.exists(test_file):
            try:
                df = pd.read_csv(test_file, usecols=["clean_code", "label"])
                df["clean_code"] = df["clean_code"].fillna("").astype(str)
                dataset = PlagiarismDataset(df, tokenizer)
                loader = DataLoader(dataset, batch_size=4, shuffle=False)
                
                metrics = evaluate_model(model, loader, device)
                results[test_file] = metrics
                
                print(f"[Seed {seed}] {test_file}:")
                print(f"  Accuracy: {metrics['accuracy']:.4f}")
                print(f"  Precision: {metrics['precision']:.4f}")
                print(f"  Recall: {metrics['recall']:.4f}")
                print(f"  F1-Score: {metrics['f1']:.4f}")
                
            except Exception as e:
                print(f"Error evaluating {test_file}: {e}")
                results[test_file] = None
        else:
            print(f"File {test_file} not found.")
            results[test_file] = None
    
    valid_results = [v for v in results.values() if v is not None]
    if valid_results:
        avg_results = {
            'accuracy': np.mean([r['accuracy'] for r in valid_results]),
            'precision': np.mean([r['precision'] for r in valid_results]),
            'recall': np.mean([r['recall'] for r in valid_results]),
            'f1': np.mean([r['f1'] for r in valid_results]),
        }
        return avg_results, results
    else:
        return None, results

def run_codellama_experiment_with_seed(seed, train_file, token, epochs=3, model_size="7b"):
    print(f"\n{'='*80}")
    print(f"RUNNING CODELLAMA EXPERIMENT WITH SEED: {seed}")
    print(f"{'='*80}")
    
    set_seed(seed)
    
    if model_size == "7b":
        model_name = "codellama/CodeLlama-7b-hf"
    elif model_size == "13b":
        model_name = "codellama/CodeLlama-13b-hf"
    elif model_size == "34b":
        model_name = "codellama/CodeLlama-34b-hf"
    else:
        model_name = "codellama/CodeLlama-7b-hf"
    
    print(f"Using model: {model_name}")
    
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        token=token,
        trust_remote_code=True
    )
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    
    train_df = pd.read_csv(train_file, usecols=["clean_code", "label"])
    train_dataset = PlagiarismDataset(train_df, tokenizer)
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
    
    model = CodeLlamaForPlagiarismDetection(model_name=model_name, token=token).to(device)
    
    optimizer = optim.AdamW(
        model.classifier.parameters(),
        lr=1e-4,
        weight_decay=0.01
    )
    criterion = nn.CrossEntropyLoss()
    
    model = train_codellama(model, train_loader, optimizer, criterion, device, epochs, seed)
    
    print(f"\n[Seed {seed}] Evaluating CodeLlama Model on test files...")
    codellama_avg_results, codellama_detailed_results = evaluate_all_test_files(model, tokenizer, device, seed)
    
    if codellama_avg_results:
        print(f"\n[Seed {seed}] CodeLlama Average Results:")
        print(f"  Accuracy: {codellama_avg_results['accuracy']:.4f}")
        print(f"  Precision: {codellama_avg_results['precision']:.4f}")
        print(f"  Recall: {codellama_avg_results['recall']:.4f}")
        print(f"  F1-Score: {codellama_avg_results['f1']:.4f}")
    
    os.makedirs("models/codellama", exist_ok=True)
    model_path = f"models/codellama/codellama_{model_size}_seed{seed}.pt"
    torch.save({
        'classifier_state_dict': model.classifier.state_dict(),
        'seed': seed,
        'model_size': model_size,
        'results': codellama_avg_results
    }, model_path)
    print(f"[Seed {seed}] CodeLlama classifier saved to: {model_path}")
    
    del model
    torch.cuda.empty_cache()
    gc.collect()
    
    return {
        'seed': seed,
        'model_size': model_size,
        'codellama_avg_results': codellama_avg_results,
        'codellama_detailed_results': codellama_detailed_results
    }

def create_results_visualizations(results_data, save_dir, model_name="CodeLlama"):
    seeds = list(results_data.keys())
    accuracies = [results_data[seed]['accuracy'] for seed in seeds]
    f1_scores = [results_data[seed]['f1'] for seed in seeds]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].plot(seeds, accuracies, 'o-', linewidth=2, markersize=8, color='purple')
    axes[0].set_xlabel('Seed', fontsize=12)
    axes[0].set_ylabel('Accuracy', fontsize=12)
    axes[0].set_title(f'{model_name} - Accuracy Across Seeds', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xticks(seeds)
    
    axes[1].plot(seeds, f1_scores, 's-', linewidth=2, markersize=8, color='green')
    axes[1].set_xlabel('Seed', fontsize=12)
    axes[1].set_ylabel('F1-Score', fontsize=12)
    axes[1].set_title(f'{model_name} - F1-Score Across Seeds', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xticks(seeds)
    
    metrics_data = [accuracies, f1_scores]
    labels = ['Accuracy', 'F1-Score']
    bp = axes[2].boxplot(metrics_data, labels=labels, patch_artist=True)
    
    colors = ['lightblue', 'lightgreen']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    
    axes[2].set_ylabel('Score', fontsize=12)
    axes[2].set_title(f'{model_name} - Metrics Distribution', fontsize=14, fontweight='bold')
    axes[2].grid(True, alpha=0.3, axis='y')
    
    for i, data in enumerate(metrics_data):
        axes[2].plot(i+1, np.mean(data), 'r_', markersize=15, markeredgewidth=2)
        axes[2].text(i+1, np.mean(data) + 0.01, f'{np.mean(data):.3f}', 
                    ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    fig_path_png = os.path.join(save_dir, f'{model_name.lower()}_results_{timestamp}.png')
    fig_path_pdf = os.path.join(save_dir, f'{model_name.lower()}_results_{timestamp}.pdf')
    
    plt.savefig(fig_path_png, dpi=300, bbox_inches='tight')
    plt.savefig(fig_path_pdf, bbox_inches='tight')
    plt.close()
    
    print(f"\nVisualizations saved to:")
    print(f"  {fig_path_png}")
    print(f"  {fig_path_pdf}")
    
    return fig_path_png

def main():
    print("="*80)
    print("CODELLAMA - PLAGIARISM DETECTION EXPERIMENT")
    print("="*80)
    
    train_file = "Train.csv"
    
    if not os.path.exists(train_file):
        print(f"Error: Train file '{train_file}' not found!")
        print("Please ensure Train.csv exists in the current directory.")
        return
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nUsing device: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
    token = authenticate_huggingface()
    if token is None:
        print("Failed to authenticate with Hugging Face. Exiting.")
        return
    
    print("\n" + "="*80)
    print("CODELLAMA MODEL SELECTION")
    print("="*80)
    print("Available CodeLlama models:")
    print("1. CodeLlama-7b (7 billion parameters)")
    print("2. CodeLlama-13b (13 billion parameters)")
    print("3. CodeLlama-34b (34 billion parameters)")
    
    model_choice = input("\nSelect model size (1, 2, or 3): ").strip()
    if model_choice == "1":
        model_size = "7b"
    elif model_choice == "2":
        model_size = "13b"
    elif model_choice == "3":
        model_size = "34b"
    else:
        print("Invalid choice. Using CodeLlama-7b (default).")
        model_size = "7b"
    
    if model_size in ["13b", "34b"] and torch.cuda.is_available():
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        if model_size == "13b" and gpu_memory < 24:
            print(f"\n⚠️  Warning: CodeLlama-13b typically needs 24+ GB GPU memory.")
            print(f"Your GPU has {gpu_memory:.1f} GB. This might cause memory issues.")
            proceed = input("Continue anyway? (y/n): ").strip().lower()
            if proceed != 'y':
                print("Switching to CodeLlama-7b.")
                model_size = "7b"
        elif model_size == "34b" and gpu_memory < 48:
            print(f"\n⚠️  Warning: CodeLlama-34b typically needs 48+ GB GPU memory.")
            print(f"Your GPU has {gpu_memory:.1f} GB. This will likely cause memory issues.")
            proceed = input("Continue anyway? (y/n): ").strip().lower()
            if proceed != 'y':
                print("Switching to CodeLlama-7b.")
                model_size = "7b"
    
    DEFAULT_SEEDS = [42, 123, 456, 789, 999, 111, 222, 333, 444, 555]
    
    print(f"\nAvailable default seeds: {DEFAULT_SEEDS}")
    seeds_input = input(f"Enter seeds to run (comma-separated, press Enter for first 2): ").strip()
    if seeds_input:
        try:
            seeds_to_run = [int(s.strip()) for s in seeds_input.split(',')][:3]
        except:
            print("Invalid input. Using first 2 default seeds.")
            seeds_to_run = DEFAULT_SEEDS[:2]
    else:
        seeds_to_run = DEFAULT_SEEDS[:2]
    
    try:
        epochs = int(input("\nNumber of training epochs for CodeLlama (recommended: 2-3): ") or "2")
    except:
        epochs = 2
    
    print(f"\n{'='*80}")
    print("EXPERIMENT CONFIGURATION")
    print("="*80)
    print(f"  Model: CodeLlama-{model_size}")
    print(f"  Seeds: {seeds_to_run}")
    print(f"  Epochs: {epochs}")
    print(f"  Batch size: 4")
    
    confirm = input("\nProceed with CodeLlama experiments? (y/n): ").strip().lower()
    if confirm != 'y':
        print("Experiment cancelled.")
        return
    
    model_dir = f"models/codellama_{model_size}"
    results_dir = f"results/codellama_{model_size}"
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(results_dir, exist_ok=True)
    
    print(f"\n{'='*80}")
    print(f"STARTING CODELLAMA EXPERIMENTS")
    print(f"{'='*80}")
    
    all_codellama_results = {}
    codellama_accuracies = []
    codellama_f1_scores = []
    
    start_time = datetime.now()
    
    for i, seed in enumerate(seeds_to_run, 1):
        print(f"\n[{i}/{len(seeds_to_run)}] ", end="")
        result = run_codellama_experiment_with_seed(seed, train_file, token, epochs, model_size)
        
        if result['codellama_avg_results']:
            all_codellama_results[seed] = result['codellama_avg_results']
            codellama_accuracies.append(result['codellama_avg_results']['accuracy'])
            codellama_f1_scores.append(result['codellama_avg_results']['f1'])
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        interim_results = []
        for s, res in all_codellama_results.items():
            interim_results.append({
                'seed': s,
                'model_size': model_size,
                'accuracy': res['accuracy'],
                'precision': res['precision'],
                'recall': res['recall'],
                'f1': res['f1']
            })
        
        if interim_results:
            interim_df = pd.DataFrame(interim_results)
            interim_path = f"{results_dir}/codellama_interim_results_{timestamp}.csv"
            interim_df.to_csv(interim_path, index=False)
            print(f"[Progress] CodeLlama interim results saved to: {interim_path}")
    
    end_time = datetime.now()
    total_time = (end_time - start_time).total_seconds() / 60
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    if all_codellama_results:
        final_results = []
        for seed, res in all_codellama_results.items():
            final_results.append({
                'seed': seed,
                'model_size': model_size,
                'accuracy': res['accuracy'],
                'precision': res['precision'],
                'recall': res['recall'],
                'f1': res['f1']
            })
        
        results_df = pd.DataFrame(final_results)
        results_path = f"{results_dir}/codellama_final_results_{timestamp}.csv"
        results_df.to_csv(results_path, index=False)
        
        print(f"\n{'='*80}")
        print("CODELLAMA RESULTS SUMMARY")
        print(f"{'='*80}")
        print(f"Experiments completed in {total_time:.2f} minutes")
        print(f"Number of seeds: {len(seeds_to_run)}")
        print(f"Model: CodeLlama-{model_size}")
        
        if codellama_accuracies:
            print(f"\nCodeLlama Average Accuracy:")
            print(f"  Mean: {np.mean(codellama_accuracies):.4f} ± {np.std(codellama_accuracies):.4f}")
            print(f"  Range: [{np.min(codellama_accuracies):.4f}, {np.max(codellama_accuracies):.4f}]")
        
        if codellama_f1_scores:
            print(f"\nCodeLlama Average F1-Score:")
            print(f"  Mean: {np.mean(codellama_f1_scores):.4f} ± {np.std(codellama_f1_scores):.4f}")
            print(f"  Range: [{np.min(codellama_f1_scores):.4f}, {np.max(codellama_f1_scores):.4f}]")
        
        print(f"\n📊 CodeLlama results saved to: {results_path}")
        
        print(f"\n{'='*80}")
        print("CODELLAMA PER-SEED RESULTS")
        print(f"{'='*80}")
        print(f"{'Seed':<8} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
        print(f"{'-'*60}")
        
        for seed in seeds_to_run:
            if seed in all_codellama_results:
                res = all_codellama_results[seed]
                print(f"{seed:<8} {res['accuracy']:<12.4f} {res['precision']:<12.4f} "
                      f"{res['recall']:<12.4f} {res['f1']:<12.4f}")
        
        create_results_visualizations(all_codellama_results, results_dir, f"CodeLlama-{model_size}")
        
        results_summary = {
            'experiment_date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'model': f"CodeLlama-{model_size}",
            'seeds_tested': seeds_to_run,
            'total_training_time_minutes': round(total_time, 2),
            'accuracy_mean': float(np.mean(codellama_accuracies)),
            'accuracy_std': float(np.std(codellama_accuracies)),
            'f1_mean': float(np.mean(codellama_f1_scores)),
            'f1_std': float(np.std(codellama_f1_scores)),
            'per_seed_results': all_codellama_results
        }
        
        summary_path = f"{results_dir}/experiment_summary_{timestamp}.json"
        with open(summary_path, 'w') as f:
            json.dump(results_summary, f, indent=4)
        
        print(f"\n📋 Experiment summary saved to: {summary_path}")
    
    print(f"\n{'='*80}")
    print("EXPERIMENT COMPLETE")
    print(f"{'='*80}")
    print(f"Total time: {total_time:.2f} minutes")
    print(f"CodeLlama classifiers saved in: {model_dir}/")
    print(f"CodeLlama results saved in: {results_dir}/")

if __name__ == "__main__":
    main()